# Steps 7 & 8: Anomaly Segmentation Evaluation

This notebook evaluates both **Pixel-based** (ERFNet) and **Mask-based** (EoMT) models for anomaly segmentation across several benchmarks, including SMIYC and Fishyscapes.

## Objectives:
1. **Evaluate ERFNet (Pixel-based):** MSP, MaxLogit, and Max Entropy.
2. **Evaluate EoMT (Mask-based):** MSP, MaxLogit, Max Entropy, and RbA across 3 checkpoints.
3. **Temperature Scaling Search:** Optimize MSP scores using the cached "Smart Trick" logic.
4. **Generate Results Tables:** Produce the exact tables requested in the project guide.

In [1]:
!pip install ood_metrics lightning gitignore_parser > /dev/null
!pip install -U 'jsonargparse[signatures]>=4.27.7' > /dev/null

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cufflinks 0.17.3 requires pandas>=0.19.2, which is not installed.
seaborn 0.13.2 requires pandas>=1.2, which is not installed.
spreg 1.9.0 requires pandas, which is not installed.
shap 0.51.0 requires pandas, which is not installed.
holoviews 1.22.1 requires pandas>=1.3, which is not installed.
arviz 0.22.0 requires pandas>=2.1.0, which is not installed.
mlxtend 0.23.4 requires pandas>=0.24.2, which is not installed.
esda 2.9.0 requires pandas>=2.1, which is not installed.
pysal 25.7 requires pandas>=1.4, which is not installed.
access 1.1.10.post3 requires pandas>=2.1.0, which is not installed.
bigframes 2.40.0 requires pandas>=1.5.3, which is not installed.
gradio 5.50.0 requires pandas<3.0,>=1.0, which is not installed.
yfinance 0.2.66 requires pandas>=1.3.0, which is not installed.
bqplot 0.12.47 requires pand

In [4]:
!pip install numpy==2.0.0 --force-reinstall
!pip install pandas --force-reinstall

  Using cached numpy-2.0.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
Using cached numpy-2.0.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cufflinks 0.17.3 requires pandas>=0.19.2, which is not installed.
seaborn 0.13.2 requires pandas>=1.2, which is not installed.
spreg 1.9.0 requires pandas, which is not installed.
shap 0.51.0 requires pandas, which is not installed.
holoviews 1.22.1 requires pandas>=1.3, which is not installed.
arviz 0.22.0 requires pandas>=2.1.0, which is not installed.
mlxtend 0.23.4 requires pandas>=0.24.2, which is not installed.
esda 2.9.0 requires pandas>=2.1, which is not installed.

  Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
ERROR: Operation cancelled by user
^C


### Resolving Dependency Conflicts

There were dependency conflicts with `numpy`, `pandas`, and `ood_metrics`. The `ood_metrics` library requires `numpy<2.0`, while `numpy==2.0.0` was installed, and `pandas` installation was interrupted. The following cells will reinstall these packages to ensure compatibility and stability for the rest of the notebook.

In [6]:
# Reinstall numpy to a version compatible with ood_metrics
# ood_metrics requires numpy<2.0
!pip install numpy==1.26.4 --force-reinstall

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.0
    Uninstalling numpy-2.0.0:
      Successfully uninstalled numpy-2.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cufflinks 0.17.3 requires pandas>=0.19.2, which is not installed.
seaborn 0.13.2 requires pandas>=1.2, which is not installed.
spreg 1.9.0 requires pandas, which is not installed.
shap 0.51.0 requires pandas, which is not installed.
holoviews 1.22.1 requires pandas>=1.3, which is not installed.
arviz 0.22.0 requires pandas>=2.1.0, which is not installed.
mlxtend 0.23.4 requires pandas>=0.24.2, which is not installed.
esda 2.9.0 requires pandas>=2.1, which is not installed.


In [ ]:
# Reinstall ood_metrics and jsonargparse after fixing numpy
!pip install ood_metrics lightning gitignore_parser > /dev/null
!pip install -U 'jsonargparse[signatures]>=4.27.7' > /dev/null

ERROR: Operation cancelled by user
^C


In [2]:
# Reinstall pandas to a version compatible with google-colab
!pip install pandas==2.2.2 --force-reinstall

  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 110.5 MB/s eta 0:00:00
Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.1/510.1 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 36.8 MB/s eta 0:00:00
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successfully uninstalled pytz-2025.2
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2026.2
    Uninstalling tzdata-2026.2:
      Successfully uninstalled

In [2]:
import os, sys
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
PROJECT_ROOT = '/content/drive/MyDrive/FundGitHubProject'
for p in (PROJECT_ROOT, os.path.join(PROJECT_ROOT, 'eomt')):
    if p not in sys.path:
        sys.path.insert(0, p)

if not os.path.exists('/content/Fundamental_Project'):
    os.symlink(PROJECT_ROOT, '/content/Fundamental_Project')

os.chdir(PROJECT_ROOT)

from posthoc_metrics import (
    get_pixel_msp, get_pixel_max_logit, get_pixel_entropy,
    get_mask_msp, get_mask_max_logit, get_mask_entropy, get_mask_rba,
    compute_metrics, cache_model_outputs, fast_temperature_search,
)
from eomt.checkpoint_utils import get_finetuned_model, get_eomt_cityscape, get_erfnet_model

# Anomaly validation datasets — downloaded separately into eval/Validation_Dataset/
# on Colab.  See README for download instructions.
from eval.Validation_Dataset import anomaly_datasets

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

/usr/local/lib/python3.12/dist-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: module 'numpy._globals' has no attribute '_signature_descriptor' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


ImportError: cannot load module more than once per process

ImportError: cannot load module more than once per process

ImportError: numpy._core.multiarray failed to import

In [ ]:
DATASETS = {
    'SMIYC RA-21':  'RoadAnomaly21',
    'SMIYC RO-21':  'RoadObstacle21',
    'FS L&F':       'FS_LostFound',
    'FS Static':    'FS_Static',
    'Road Anomaly': 'RoadAnomaly',
}

dataloaders = {}
for name, key in DATASETS.items():
    dm = anomaly_datasets.AnomalyDataModule(dataset_name=key, img_size=(640, 640))
    dm.setup()
    dataloaders[name] = dm.val_dataloader()
    print(f'  {name}: {len(dataloaders[name])} batches')


In [ ]:
# Paths to model weights — update if your folder layout differs
CKPT = {
    'ERFNet':      'trained_models/erfnet_encoder_pretrained.pth.tar',
    'COCO':        'eomt/eomt_weights/eomt_coco.bin',
    'Cityscapes':  'eomt/eomt_weights/eomt_cityscapes.bin',
    # Fine-tuned: best checkpoint from Step 5 experiments
    'Fine-tuned':  next(
        (p for p in [
            'checkpoints/lora-all-blocks/last.ckpt',
            'checkpoints/lora-decoder-blocks/last.ckpt',
            'checkpoints/lora-head-only/last.ckpt',
        ] if os.path.exists(p)),
        None
    ),
}
print('Checkpoints:')
for k, v in CKPT.items():
    status = '✅' if v and os.path.exists(v) else '❌ missing'
    print(f'  {k}: {v}  {status}')


## Step 7 — Pixel-based baselines (ERFNet)

In [ ]:
def evaluate_pixel_model(model, dataloader, method_name, temperature=1.0):
    """Single forward pass per image; all scoring is done on the logits."""
    model.eval()
    scoring_fns = {
        'MSP':         lambda l: get_pixel_msp(l, temperature=temperature),
        'MaxLogit':    lambda l: get_pixel_max_logit(l),
        'Max Entropy': lambda l: get_pixel_entropy(l, temperature=temperature),
    }
    fn = scoring_fns[method_name]
    all_scores, all_gts = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f'ERFNet {method_name}', leave=False):
            img    = batch['image'].to(device)
            gt     = batch['label']
            logits = model(img)
            score  = fn(logits)
            if score.shape[-2:] != gt.shape[-2:]:
                score = torch.nn.functional.interpolate(
                    score.unsqueeze(1), size=gt.shape[-2:], mode='bilinear'
                ).squeeze(1)
            all_scores.append(score.cpu().numpy().flatten())
            all_gts.append(gt.numpy().flatten())

    return compute_metrics(np.concatenate(all_scores), np.concatenate(all_gts))


erfnet = get_erfnet_model(CKPT['ERFNet'], device=str(device))

pixel_results = {}
for ds_name, loader in dataloaders.items():
    pixel_results[ds_name] = {}
    for method in ['MSP', 'MaxLogit', 'Max Entropy']:
        pixel_results[ds_name][method] = evaluate_pixel_model(erfnet, loader, method)

df_pixel = pd.DataFrame(
    {(ds, m): pixel_results[ds][m] for ds in pixel_results for m in pixel_results[ds]}
).T
print(df_pixel.to_string())


## Step 8 — Mask-based baselines (EoMT)

Model outputs are **cached to disk once** per (model, dataset) pair.
All four scoring methods (MSP, MaxLogit, Max Entropy, RbA) are then computed
from the cache in seconds — no repeated GPU inference.

In [5]:
import os, torch
from posthoc_metrics import (
    get_mask_msp, get_mask_max_logit, get_mask_entropy, get_mask_rba, compute_metrics
)

CACHE_ROOT = 'eval_cache'

def get_cache_dir(model_name, ds_name):
    d = os.path.join(CACHE_ROOT, model_name.replace(' ', '_'), ds_name.replace(' ', '_'))
    os.makedirs(d, exist_ok=True)
    return d

class _TupleLoader:
    """Wraps a dict-batch loader to yield (image, label) tuples."""
    def __init__(self, loader): self.loader = loader
    def __iter__(self):
        for b in self.loader: yield b['image'], b['label']
    def __len__(self): return len(self.loader)

def score_from_cache(cache_dir, method, temperature=1.0):
    """Compute anomaly scores for one method over all cached samples."""
    scoring_fns = {
        'MSP':         lambda mc, mp: get_mask_msp(mc, mp, temperature=temperature),
        'MaxLogit':    lambda mc, mp: get_mask_max_logit(mc, mp),
        'Max Entropy': lambda mc, mp: get_mask_entropy(mc, mp, temperature=temperature),
        'RbA':         lambda mc, mp: get_mask_rba(mc, mp, temperature=temperature),
    }
    fn = scoring_fns[method]

    all_scores, all_gts = [], []
    for fname in sorted(f for f in os.listdir(cache_dir) if f.endswith('.pt')):
        data  = torch.load(os.path.join(cache_dir, fname), map_location='cpu')
        score = fn(data['mask_cls'], data['mask_pred'])
        all_scores.append(score.flatten().numpy())
        all_gts.append(data['gt'].flatten().numpy())

    import numpy as np
    return compute_metrics(np.concatenate(all_scores), np.concatenate(all_gts))


# ── Load models ──────────────────────────────────────────────────────────────
mask_models = {}
if CKPT['COCO']:        mask_models['COCO']        = get_eomt_cityscape(CKPT['COCO'],       device=str(device))
if CKPT['Cityscapes']:  mask_models['Cityscapes']   = get_eomt_cityscape(CKPT['Cityscapes'], device=str(device))
if CKPT['Fine-tuned']:  mask_models['Fine-tuned']   = get_finetuned_model(CKPT['Fine-tuned'],device=str(device))

# ── Cache outputs (skips if cache already exists) ─────────────────────────────
for model_name, model in mask_models.items():
    for ds_name, loader in dataloaders.items():
        cache_dir = get_cache_dir(model_name, ds_name)
        if not any(f.endswith('.pt') for f in os.listdir(cache_dir)):
            print(f'Caching {model_name} × {ds_name}...')
            cache_model_outputs(model, _TupleLoader(loader), cache_dir, device=str(device))
        else:
            print(f'  Cache exists: {model_name} × {ds_name} ({len(os.listdir(cache_dir))} files)')

print('\nAll outputs cached.')



--- Initializing EoMT Cityscapes Architecture ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights from: eomt/eomt_weights/eomt_coco.bin


RuntimeError: Error(s) in loading state_dict for EoMT:
	size mismatch for q.weight: copying a param with shape torch.Size([200, 768]) from checkpoint, the shape in current model is torch.Size([100, 768]).
	size mismatch for class_head.weight: copying a param with shape torch.Size([134, 768]) from checkpoint, the shape in current model is torch.Size([20, 768]).
	size mismatch for class_head.bias: copying a param with shape torch.Size([134]) from checkpoint, the shape in current model is torch.Size([20]).

In [ ]:
# ── Evaluate all methods at T=1.0 from cache ─────────────────────────────────
METHODS = ['MSP', 'MaxLogit', 'Max Entropy', 'RbA']

mask_results = {}
for model_name in mask_models:
    mask_results[model_name] = {}
    for ds_name in dataloaders:
        mask_results[model_name][ds_name] = {}
        cache_dir = get_cache_dir(model_name, ds_name)
        for method in METHODS:
            mask_results[model_name][ds_name][method] = score_from_cache(
                cache_dir, method, temperature=1.0
            )

df_mask = pd.DataFrame(
    {(mn, ds, m): mask_results[mn][ds][m]
     for mn in mask_results for ds in mask_results[mn] for m in mask_results[mn][ds]}
).T
print(df_mask.to_string())


## Temperature Scaling

Search for the best softmax temperature `T` using cached Fine-tuned logits
on Road Anomaly as calibration set.  Once `best_t` is found, **all MSP and
Max Entropy results are re-scored** with that temperature so the final table
reflects the calibrated numbers.

In [ ]:
import numpy as np

if 'Fine-tuned' not in mask_models:
    print('No fine-tuned checkpoint — skipping temperature scaling.')
    best_t = 1.0
else:
    calib_cache = get_cache_dir('Fine-tuned', 'Road Anomaly')

    temps = [0.5, 0.75, 1.0, 1.25, 1.5]
    search_results = fast_temperature_search(calib_cache, scoring_fn=get_mask_msp,
                                             temperatures=temps)

    best_t = max(search_results, key=lambda t: search_results[t]['auprc'])

    df_temp = pd.DataFrame([
        {'T': t,
         'AuPRC': f"{search_results[t]['auprc']:.4f}",
         'FPR95': f"{search_results[t]['fpr95']:.4f}",
         'Best':  '✅' if t == best_t else ''}
        for t in temps
    ])
    print(f'Best T = {best_t}')
    display(df_temp)

    # ── Re-score temperature-sensitive methods with best_t ───────────────────
    if best_t != 1.0:
        print(f'\nRe-scoring MSP and RbA with T={best_t}...')
        for model_name in mask_models:
            for ds_name in dataloaders:
                cache_dir = get_cache_dir(model_name, ds_name)
                for method in ['MSP', 'Max Entropy', 'RbA']:
                    mask_results[model_name][ds_name][f'{method} (T={best_t})'] = \
                        score_from_cache(cache_dir, method, temperature=best_t)

        # Rebuild table with calibrated columns included
        df_mask = pd.DataFrame(
            {(mn, ds, m): mask_results[mn][ds][m]
             for mn in mask_results for ds in mask_results[mn] for m in mask_results[mn][ds]}
        ).T
        print('\nUpdated results (calibrated columns appended):')
        print(df_mask.to_string())


## Results Summary

In [ ]:
print("\n=== ERFNet Pixel Results ===")
print(df_pixel.to_string())
print("\n=== EoMT Mask Results (T=1.0 + calibrated) ===")
print(df_mask.to_string())
